# manual-chain-forward-and-back — ex1: manually chain forward log/exp and run backward by hand

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `manual-chain-forward-and-back`. Running the final beacon cell reports progress against the `Backprop: manual chain forward-and-back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: manual chain forward-and-back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`manual-chain-forward-and-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "manual-chain-forward-and-back"
DD_SUBTOPIC = "Backprop: manual chain forward-and-back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Manual forward-and-back chain — quick refresher

Before the dispatcher exists, you can run a short chain by hand:

```
# Forward:  a → b = log(a) → c = exp(b)
b = log(a)
c = exp(b)

# Backward (assume dL/dc is given):
dL_db = exp_back(dL_dc, c, b)     # back_fn for c = exp(b)
dL_da = log_back(dL_db, b, a)     # back_fn for b = log(a)
```

Two patterns to internalize:
- **Reverse the call order.** The forward computed `a → b → c`; the   backward computes `dL_dc → dL_db → dL_da`. Same nodes, opposite   direction.
- **Each back_fn receives `(grad_out, out_at_that_node, *inputs)`.**   `exp_back` gets `(dL_dc, c, b)` — the cached output `c` (used to   compute the gradient) AND the input `b` it was applied to. The   uniform signature is what makes the dispatcher possible later.

### Exercise 1 — manually chain forward log/exp and run backward by hand

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the manual forward-then-backward chain by computing b = log(a), c = exp(b) on the forward pass and then dL_da via exp_back followed by log_back in the reverse order.
> Keywords: manual-chain, forward-backward, log, exp, reverse-order
> ```

**KCs targeted:** `manual-chain-forward-and-back`, `back-fn-uses-cached-out`

Implement `manual_chain(a, dL_dc)` — a hand-run forward+backward for a length-2 chain, BEFORE the dispatcher exists.

**Forward pass.**
```
b = log(a)
c = exp(b)
```

**Backward pass.** Given `dL/dc`, compute `dL/da` by running the chain in reverse:
```
dL_db = exp_back(dL_dc, c, b)
dL_da = log_back(dL_db, b, a)
```

where:
- `exp_back(grad_out, out, x) = grad_out * out`  (uses the cached `out`, since `d/dx exp(x) = exp(x) = out`)
- `log_back(grad_out, out, x) = grad_out / x`    (uses `x`, since `d/dx log(x) = 1/x`)

**Return** a 4-tuple `(b, c, dL_db, dL_da)` so the test can inspect every intermediate.

**The point of this drill** is to internalize the reverse-order pattern: forward goes `a → b → c`; backward goes `dL_dc → dL_db → dL_da`. Each back_fn takes `(grad_out, cached_out, input)` — the same signature the dispatcher will use later.

All inputs are plain `torch.Tensor`, float dtype, same shape. Assume `a > 0` (so `log(a)` is well-defined).

In [ ]:
def manual_chain(a: Tensor, dL_dc: Tensor) -> tuple:
    """Forward: b = log(a), c = exp(b). Backward: return (b, c, dL_db, dL_da)."""
    raise NotImplementedError()


def _test_ex1():
    # --- sanity: log+exp is the identity, so c == a ---
    a = t.tensor([1.0, 2.0, 4.0])
    dL_dc = t.tensor([1.0, 1.0, 1.0])
    b, c, dL_db, dL_da = manual_chain(a, dL_dc)
    assert t.allclose(c, a, atol=1e-5), f'log+exp should be identity: c={c} vs a={a}'
    assert t.allclose(b, t.log(a), atol=1e-5), f'b should be log(a): {b}'

    # --- backward shapes ---
    assert dL_db.shape == a.shape
    assert dL_da.shape == a.shape

    # --- backward values ---
    # exp_back: dL_db = dL_dc * c (and c == a)
    expected_dL_db = dL_dc * c
    assert t.allclose(dL_db, expected_dL_db, atol=1e-5), (
        f'dL_db wrong: got {dL_db}, expected {expected_dL_db}'
    )
    # log_back: dL_da = dL_db / a
    expected_dL_da = dL_db / a
    assert t.allclose(dL_da, expected_dL_da, atol=1e-5), (
        f'dL_da wrong: got {dL_da}, expected {expected_dL_da}'
    )
    # Composite check: dL_da should equal dL_dc * (c / a) = dL_dc (since c == a).
    assert t.allclose(dL_da, dL_dc, atol=1e-5), (
        f'For log+exp = identity, dL_da should equal dL_dc: {dL_da} vs {dL_dc}'
    )

    # --- non-unit dL_dc — chain rule scales each entry ---
    a = t.tensor([2.0, 3.0])
    dL_dc = t.tensor([5.0, -2.0])
    b, c, dL_db, dL_da = manual_chain(a, dL_dc)
    assert t.allclose(dL_da, dL_dc, atol=1e-5), 'composite log+exp identity'

    # --- witness against torch.autograd on the full chain ---
    a_ref = t.tensor([1.5, 2.5, 4.5], requires_grad=True)
    c_ref = t.exp(t.log(a_ref))
    loss = (c_ref * t.tensor([0.5, -1.0, 2.0])).sum()
    loss.backward()
    _, _, _, dL_da_ours = manual_chain(a_ref.detach(), t.tensor([0.5, -1.0, 2.0]))
    assert t.allclose(dL_da_ours, a_ref.grad, atol=1e-5), (
        f'chain disagrees with autograd: ours={dL_da_ours}, ref={a_ref.grad}'
    )

    # --- reverse-order invariant: dL_db must be computed BEFORE dL_da ---
    # (If you implemented forward order by mistake — log_back first — the
    # numerical mismatch would already have caught it; this is a structural
    # nudge for the explanation.)
    # Quick structural smoke check: each gradient depends on the previous one.
    # Verified by the value asserts above.
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def manual_chain(a: Tensor, dL_dc: Tensor) -> tuple:
    # --- forward: a -> b -> c ---
    b = t.log(a)
    c = t.exp(b)
    # --- backward (reverse order): dL_dc -> dL_db -> dL_da ---
    # exp_back uses the cached `out` (c): d/dx exp(x) = exp(x).
    dL_db = dL_dc * c
    # log_back uses the input x (a): d/dx log(x) = 1/x.
    dL_da = dL_db / a
    return b, c, dL_db, dL_da
```

**Why this drill matters even before the dispatcher.** Once `BACK_FUNCS` and the topo-sort exist, this whole function collapses to `out.backward()`. But the dispatcher is just AUTOMATING what you just did by hand — pick the right back_fn per node, call it with `(grad_out, cached_out, input)`, walk in reverse-topological order. Internalize the pattern at length-2 and the n-node case is the same thing in a loop.

**Why `exp_back` uses `out` and `log_back` uses `x`.** `d/dx exp(x) = exp(x) = out` — the cached forward output IS the derivative, so it's the cheapest pick. `d/dx log(x) = 1/x` — no relationship to `out = log(x)`, so we have to read `x` directly. The shared `(grad_out, out, x)` signature lets each op pick whichever cache makes sense.

**Identity check is the strongest test.** `exp(log(a)) == a`, so by the chain rule `d/da (exp(log(a))) = 1`, which means `dL/da == dL/dc` exactly. If your code passes this you almost certainly got the reverse order right.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()